In [2]:
import numpy as np
import json
import os

#### Embedder: 

I continue to adapt the embedder. I would like to make something myself, but for now the SentenceTransformer package is seeming to work the best.

In [3]:
'''
this is version 1 of the embedder:

def alphebet_embeder(text):
    text = text.lower()
    vector = np.zeros(26) # one space for each letter
    for char in text:
        if char.isalpha():
            index = ord(char) - ord('a') # makes a=1, b=2, etc
    # we now normalize the vectors so that in the future
    # we can calculate cosine similarity with only a dot product.
    norm = np.linalg.norm(vector)
    if norm == 0: # avoid division by zero
        return vector
    return vector / norm
'''

"\nthis is version 1 of the embedder:\n\ndef alphebet_embeder(text):\n    text = text.lower()\n    vector = np.zeros(26) # one space for each letter\n    for char in text:\n        if char.isalpha():\n            index = ord(char) - ord('a') # makes a=1, b=2, etc\n    # we now normalize the vectors so that in the future\n    # we can calculate cosine similarity with only a dot product.\n    norm = np.linalg.norm(vector)\n    if norm == 0: # avoid division by zero\n        return vector\n    return vector / norm\n"

In [4]:
# here i present version 2, pulling in from an external library called SentenceTransformer

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_v2(text):
    return model.encode(text, normalize_embeddings=True)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8051.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def embed(text):
    return embed_v2(text)

#### Save and Load: 

we are going to save vectors and their text as JSON files. we'll call this below in Add and Search, so these are mostly helper functions.
In Version 2 (the one you see below the plain-text v1, I use numpy arrays instead of a string representation of vectors).

In [6]:
'''
DB_FILE = "mydb.json"

def load_db():
    if not os.path.exists(DB_FILE):
        return []
    with open(DB_FILE, "r") as file:
        raw = json.load(file)
    for entry in raw:
        entry["vector"] = np.array(entry["vector"])
    return raw

def save_db(db):
    raw = []
    for entry in db:
        raw.append({"text": entry["text"], 
                    "vector": entry["vector"].tolist()}) # fixes the json problem
    with open(DB_FILE, "w") as file:
        json.dump(raw, file, indent=2)
'''

'\nDB_FILE = "mydb.json"\n\ndef load_db():\n    if not os.path.exists(DB_FILE):\n        return []\n    with open(DB_FILE, "r") as file:\n        raw = json.load(file)\n    for entry in raw:\n        entry["vector"] = np.array(entry["vector"])\n    return raw\n\ndef save_db(db):\n    raw = []\n    for entry in db:\n        raw.append({"text": entry["text"], \n                    "vector": entry["vector"].tolist()}) # fixes the json problem\n    with open(DB_FILE, "w") as file:\n        json.dump(raw, file, indent=2)\n'

In [7]:

DB_VECS = "vectors.npy"
DB_TEXTS = "texts.json"


def load_db():
    vectors = np.load(DB_VECS) if os.path.exists(DB_VECS) else np.empty((0,     384))
    text = json.load(open(DB_TEXTS)) if os.path.exists(DB_TEXTS) else []
    return vectors, text

def save_db(vecs, text):
    np.save(DB_VECS, vecs)
    json.dump(text, open(DB_TEXTS, 'w'))

#### Add: 

Embed a piece of text and store it. In version 2, I am changing the add function to append, and writing a new add function that check to see if it is different from what has already been input. If the cosine similarity is greater than a certain amount, then we will ask if you really want to add the new information or not.

In [8]:
def threshold_check(vectors, new_vector, texts, threshold=.95):
    scores = vectors @ new_vector
    top = np.argsort(scores)[::-1] # most of this was first used in the search() function

    similar = []
    for i, j in enumerate(top):
        if scores[j] > threshold:
            similar.append(texts[j])
    
    if not similar:
        return True

    similar_string = '\n\n'.join(similar)
    confirm = input(f"the database is already storing the following similar information: '{similar_string}' are you sure you want to add this information? y/n")

    if confirm == 'y':
        return True


def add(text):
    vectors, texts = load_db()
    vector = embed(text)
    # above is an interesting line. Either we add to our matrix or we essentially reshape our vector into a matrix
    if threshold_check(vectors, vector, texts):  # check before appending the new vector
        vectors = np.vstack([vectors, vector]) if len(vectors) else vector.reshape(1,-1)
        texts.append(text)
        save_db(vectors, texts)
        print(f"Added: '{text}'")

#### Search: two steps. 

(1) embed query (2) find cosine similarity with all other vectors. I did some cool things here! When I started, I had a list of vectors in a json file which forced me to loop through each vector and find the dot product of it with the query vector in order to find cosine similarity. This required a for loop. Now, because I am putting all of my vectors into a matrix, I can just use matrix multiplication and do all of the calculation at once. Because each row of the matrix is a normalized vector, I end up with a new matrix that shows me the cosine similarity of a bunch of vectors to the query vector.

In [9]:
def cosine_similarity(a,b):
    return np.dot(a, b)

def search(query, num_out=1):
    vectors, texts = load_db()
    if len(vectors) == 0:
        print("Database is empty.")
        return
    
    query_vector = embed(query)
    scores = vectors @ query_vector # cool part here! see markdown above.
    top = np.argsort(scores)[::-1][:num_out] # this is confusing. It takes the place-value of the scores and then flips because we want greatest to least. and
    
    print()
    print(f"Search results for: '{query}'")
    print("-" * 40)
    for i in top:
        print(f"{i}. [{scores[i]:.3f}] {texts[i]}")
    print()

some tests:

In [10]:
'''
add("I love hiking in the mountains")
add("Python is a great programming language")
add("The sunset over the ocean was beautiful")
add("I really love to study")
add("Camping under the stars is peaceful")
'''


'\nadd("I love hiking in the mountains")\nadd("Python is a great programming language")\nadd("The sunset over the ocean was beautiful")\nadd("I really love to study")\nadd("Camping under the stars is peaceful")\n'

In [11]:
'''
search("time when I was happy about a great date")
'''

'\nsearch("time when I was happy about a great date")\n'

How to clear the database:

In [12]:
def eraser():
    confirmation = input("Proceed to clear database? y/n: ")
    if confirmation == 'y':
        code = input("type 'I confirm I want to erase all data' exactly to confirm: ")
        if code == "I confirm I want to erase all data":
            save_db(np.empty((0, 384)), [])
            for filename in os.listdir('text_entries'):
                file_path = os.path.join('text_entries', filename)
                if os.path.isfile(file_path):
                    os.remove(file_path)
            print("Database and text_entries folder cleared.")
        else:
            print("wrong code. please try again")
    return

eraser()

Database and text_entries folder cleared.


# This one is Huge!

Here I am going to embed my journal and it will then be queriable. This is just a demo for what I plan to do with my dad's 25 years of daily journaling

the first problem here is that all my journal entries were downloaded as html files with a TON of information about what the look like
and generally just stuff I do not care about. I first need to go into the HTML files, parse them, and pull out just the text. That's all I care for here.
since i am going to have to do this for many files, I am going to have to learn a bit about how to work in entire folders.

In [13]:
from bs4 import BeautifulSoup
import os


def html_to_text(file_path):
    with open(file_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    text = soup.get_text()
    out_name = file_path.split('/')[-1].replace(".html", ".txt")
    text = " ".join(text.split())
    with open(os.path.join("text_entries", out_name), "w") as f:
        f.write(text)



def folder_converter(folder):
    for filename in sorted(os.listdir(folder)):
        if filename.endswith(".html"):
            html_to_text(os.path.join(folder, filename))


if __name__ == '__main__':
    folder = "AppleJournalEntries/Entries"
    folder_converter(folder)

##### and now for the fun part

In [14]:
import os

folder = 'text_entries'
files = sorted(os.listdir(folder))

for filename in files:
    if filename.endswith(".txt"):
        path = os.path.join(folder, filename)
        with open(path) as f:
            text = f.read().strip()
        print(f"--- {filename} ---")
        add(text)

--- 2025-06-02.txt ---
Added: 'Sunday, June 1, 2025 After church max and I changed in the chapel and left for Seattle to go see grandma and grandpa Calvert. The whole drive, I felt melancholy, max and I were talking about the time passing too quickly. Mostly, I felt that weird feeling because I couldn’t shake the face Sean made when I told him we were going to Seattle for the next couple days. He just looked sad. Maybe he just wanted to see grandpa, but I felt like he was sad that I was leaving. Sean doesn’t know how much of an influence he has on me. That boy means so much to me. It makes me feel different about all of it, about going to college, about everything. I just want to jump on the trampoline with him. I think a little bit of what I feel is a little bit of a fatherly connection. I guess I know I will feel this way when I have to leave my kids. Dang, I felt really deeply today.'
--- 2025-06-03.txt ---
Added: 'Monday, June 2, 2025 Today was slow and kind of perfect. We woke up 

# Let's Build a Better Query Function

In [15]:
import os

folder = 'text_entries'

def search_v2(num_out=1):
    year = input('what year are you looking into?: ')
    month = input('month?: ')
    query = input('tell me a little bit about the experience: ')

    query_vector = embed(query)
    vectors, texts = load_db()
    scores = {}

    filenames = [f for f in sorted(os.listdir(folder)) if f.endswith('.txt')]

    for i, filename in enumerate(filenames):
        date = filename.split('-')
        if date[0] == year:
            if date[1] == month:
                score = query_vector @ vectors[i]
                scores[i] = score

    for i in sorted(scores, key=lambda i: scores[i], reverse=True)[:num_out]:
        filepath = os.path.join(folder, files[i])
        with open(filepath, 'r') as f:
            print(f.read())

search_v2()


Sunday, January 11, 2026 Ok, in the last week I spent 4 days in Santiago with my dad, had 3 days of class, went to the ER 2 times and ate 1 e.coli infested meal. It was a crazy week but I feel great now. I also skied on Thursday and yesterday, pushing myself way more than I ever had before. Great few days.


I would like to add something to the way this works. I wish I could prioritize people as part of the journal. I may forget some things about what I did, but there are key words that are more important than others. I wonder if I could somehow embed certain people into the model such that the mention of a specific person or place strongly increases the likelyhood of the journal entry being selected. How would I go about doing this?